In [ ]:
import os, json
import torch
import transformers
import pandas as pd
import numpy as np

import transformers 
from transformers import AutoTokenizer

import MeMoHF
from MeMoHF.modelling_memo_tokenizer import MeMoTokenizer
from MeMoHF.modelling_memo_configuration import MeMoConfig
from MeMoHF.modelling_memo import MeMoForCausalLM
from MeMoHF.evaluating_memo import Evaluation
from MeMoHF.utils import (
    seed_everything,
    load_model_and_tokenizer,
    save_data_to_disk,
    load_from_disk
)

import datasets 
from datasets import Dataset, DatasetDict, Features, Value, load_dataset, load_from_disk, concatenate_datasets

os.environ['CUDA_VISIBLE_DEVICES'] = '0'



In [ ]:
from learning_evaluation import create_large_sample

In [ ]:
data_path = '/home/davide/.cache/huggingface/datasets/wikipedia/20200501.en/1.0.0/009f923d9b6dd00c00c8cdc7f408f2b47f45dd4f5fb7982a21f9448f4afbe475/wikipedia-train.arrow'
data = dict(
    train=Dataset.from_file(data_path)
)

In [ ]:
data['train'].select_columns('text')

In [ ]:
import learning_evaluation
from learning_evaluation import create_all_datasets

# create_all_datasets(main_data_dir='training_data')


In [ ]:
def convert_text_into_cfg(text):
    cfg_list = [
        (param.split('=['))
        for param in text.split(']-')
    ]
    return {
        param[0]:param[1]
        for param in cfg_list
    }

convert_text_into_cfg(os.path.basename('bla/di/bla/max_length=[1024]-d=[1024]-l=[4]-h=[4]-batch_size=[64]-save_every_k_batches=[3]-data_name=[n=1000]-seed=[42]-batch_id=[3]'))

In [ ]:
import pandas as pd

# Sample DataFrame
df = pd.DataFrame([
    {'lr': 0.01, 'batch_size': 32, 'optimizer': 'adam'},
    {'lr': 0.001, 'batch_size': 64, 'optimizer': 'sgd'},
    {'lr': 0.01, 'batch_size': 64, 'optimizer': 'adam'},
])

# Dictionary with a subset of keys
partial_experiment = {'lr': 0.01, 'optimizer': 'adam'}

# Subset DataFrame to the relevant columns and compare
match = (df[partial_experiment.keys()] == pd.Series(partial_experiment)).all(axis=1)

# Check if any row matches the subset
exists = match.any()

print("Subset match exists:", exists)

In [ ]:
import pandas as pd

# Existing DataFrame
df = pd.DataFrame([
    {'lr': 0.01, 'batch_size': 32, 'optimizer': 'adam'},
    {'lr': 0.001, 'batch_size': 64, 'optimizer': 'sgd'},
])

# New row as a dictionary
new_row = {'lr': 0.005, 'batch_size': 128, 'optimizer': 'adam'}

# Add the new row
df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

print(df)

Experiments with sequence view augmentation for computing multiple sequences in parallel given [batch_size] texts

In [ ]:
import torch 
import torch.nn.functional as F

# Example input
batch_size = 3
seq_len = 4
hidden_dim = 2
window_size = 2

# x = torch.randn(batch_size, seq_len, hidden_dim)  # (B, S, H)
x = torch.tensor(
    [
        [[1,2],[3,4],[5,6],[7,8]],
        [[9,10],[11,12],[13,14],[15,16]],
        [[17,18],[19,20],[21,22],[23,24]],
    ],
    dtype=torch.int32
)
y = torch.tensor(
    [
        [1,2,3,4],
        [5,6,7,8],
        [9,10,11,12],
    ],
    dtype=torch.int32
)
print(x)
print(y)

In [ ]:
num_windows = seq_len - window_size + 1 
x_windows = x.permute(0, 2, 1) # (B, H, S)
# Now unfold sequence dimension (dim=2)
x_windows = x_windows.unfold(dimension=2, size=window_size, step=1) # (B, H, num_windows, window_size)
# Now bring it back to (B, num_windows, window_size, H)
x_windows = x_windows.permute(0, 2, 3, 1) # (B, num_windows, window_size, H)
# Reshape to (B * num_windows, window_size, H)
x_windows = x_windows.contiguous().view(-1, window_size, hidden_dim)
print(x_windows)

y_windows = y[:, 1:].unfold(dimension=1, size=1, step=1)
y_windows = y_windows.contiguous().view(-1, 1)
print(y_windows)

In [ ]:
new_hidden_dim = hidden_dim
final_x = x_windows.sum(dim=1)
final_x = final_x.view(batch_size, num_windows, new_hidden_dim)
final_y = y_windows.view(batch_size, -1)

print(final_x)
print(final_y)

In [2]:
0

0

MEMO Debugging

In [1]:
import torch
from MeMoHF.modelling_memo_tokenizer import MeMoTokenizer
from MeMoHF.modelling_memo_configuration import MeMoConfig
from MeMoHF.modelling_memo import MeMoForCausalLM
from MeMoHF.evaluating_memo import Evaluation
from MeMoHF.utils import seed_everything

seed_everything(42)

# d, h, l = 1024, 4, 4
# chunk_length = 256

# d, h, l = 1024, 4, 2
# chunk_length = 16
d,h,l = 2048, 4, 2
chunk_length =  (h ** l)

# Initializing a standard Tokenizer
max_length = chunk_length 
tokenizer = MeMoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", 
                                          padding_side='left', truncation_side='left', 
                                          model_max_length=max_length, head_number=h)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id

# Intializing Memo Configuration
config = MeMoConfig(vocab_size=len(tokenizer), #tokenizer.vocab_size, 
    hidden_size=d, 
    num_hidden_layers=l,
    num_attention_heads=h,
    chunk_length=chunk_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    compositionOp='prod',
    padding_seq_idx=tokenizer.pad_token_id,
    padding_vector_component_values=1/(d**(1/2))  #new!
                    
)

# Initializing the Memo Model from the configuration

model = MeMoForCausalLM(config) 
model.training = True
model.train()

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    model.to('cuda')


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPTNeoXTokenizer'. 
The class this function is called from is 'MeMoTokenizer'.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Setting pad token and pad token id = <|endoftext|>, 0
Padding input tokens in MeMo architecture:  0.022097086912079608
MeMoEmbedding 0.022097086912079608
MeMoEmbedding 0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0.022097086912079608
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0.022097086912079608
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
GPU: NVIDIA RTX A6000 is available.


In [2]:
model.memo.output_encoder.forward(torch.tensor([model.memo.output_encoder.padding_idx]))

tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0',
       grad_fn=<EmbeddingBackward0>)

In [3]:
display(model.memo.encoder.weight[model.memo.encoder.padding_seq_idx], 
model.memo.output_encoder.weight[model.memo.encoder.padding_seq_idx])

tensor([0.0221, 0.0221, 0.0221,  ..., 0.0221, 0.0221, 0.0221], device='cuda:0',
       grad_fn=<SelectBackward0>)

tensor([0., 0., 0.,  ..., 0., 0., 0.], device='cuda:0',
       grad_fn=<SelectBackward0>)

In [4]:
model.memo.encoder.weight[model.memo.encoder.padding_idx], model.memo.output_encoder.weight[model.memo.encoder.padding_idx]

(tensor([0.0221, 0.0221, 0.0221,  ..., 0.0221, 0.0221, 0.0221], device='cuda:0',
        grad_fn=<SelectBackward0>),
 tensor([0., 0., 0.,  ..., 0., 0., 0.], device='cuda:0',
        grad_fn=<SelectBackward0>))

In [5]:
model

MeMoForCausalLM(
  (memo): MeMo(
    (encoder): MeMoEmbedding(50277, 2048, padding_idx=0)
    (output_encoder): MeMoEmbedding(50277, 2048, padding_idx=0)
    (layers): MeMoLayers(
      (0): MeMoLayer(
        (CMM_OUT): CorrelationMatrixMemory(in_features=2048, out_features=2048)
      )
      (1): MeMoLayer(
        (CMM): CorrelationMatrixMemory(in_features=2048, out_features=2048)
        (CMM_OUT): CorrelationMatrixMemory(in_features=2048, out_features=2048)
      )
    )
  )
  (lm_head): MeMoEmbedding(50277, 2048, padding_idx=0)
)

In [6]:
#my_first_text = "b c d e f a b c d e f"
my_first_text = "a b c d e f g h i l m"
my_second_text = "j k m a b c" # "n o p q r s t"#
memo_input_1 = tokenizer.get_text_batch_encoding([my_first_text]*1)  # Writing the same doc 8 times to stress the memorization with batch
memo_input_2 = tokenizer.get_text_batch_encoding([my_second_text]*1) # Writing the same doc 8 times to stress the memorization with batch

print(tokenizer.decode(memo_input_1['input_ids'][0]))
print(tokenizer.decode(memo_input_1['labels'][0]))

print(memo_input_1['input_ids'], len(memo_input_1['input_ids'][0]))
print(memo_input_1['labels'], len(memo_input_1['input_ids'][0]))

<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>a b c d e f g h i l
<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>a b c d e f g h i l m
tensor([[  0,   0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288,
         891, 298]]) 16
tensor([[  0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288, 891,
         298, 278]]) 16


In [7]:
memo_input_1

{'input_ids': tensor([[  0,   0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288,
          891, 298]]),
 'labels': tensor([[  0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288, 891,
          298, 278]])}

In [8]:
model.memo.encoder.padding_seq_idx, model.memo.encoder.padding_idx

(0, 0)

In [9]:
enc = model.memo.encoder.encode(memo_input_1['input_ids'])
dec = model.memo.encoder.decode(enc)
print(dec)

tokenizer.batch_decode(dec[0])

(tensor([[  0,   0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288,
         891, 298]], device='cuda:0'), tensor([[1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0370, 0.9897, 1.0047,
         0.9610, 1.0206, 0.9835, 1.0508, 0.9744, 1.0036, 1.0072]],
       device='cuda:0', grad_fn=<MaxBackward0>))


['<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>a b c d e f g h i l']

In [10]:
model.memo.encoder.weight[270].T @ model.memo.encoder.weight[270] 

/tmp/ipykernel_3946142/3403464107.py:1: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3697.)
  model.memo.encoder.weight[270].T @ model.memo.encoder.weight[270]


tensor(0.9897, device='cuda:0', grad_fn=<DotBackward0>)

In [11]:
a = tokenizer.get_text_batch_encoding_for_loss(text=[my_first_text])
for k in a:
    print(k, a[k].shape)
    print(a[k])

input_ids torch.Size([1, 16])
tensor([[  0,   0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288,
         891, 298]])
labels torch.Size([1, 16])
tensor([[-100, -100, -100, -100, -100,   66,  270,  260,  277,  299,  269,  305,
          288,  891,  298,  278]])


In [12]:
# evaluation method
from MeMoHF.evaluating_memo import EvaluationUpdateNew
evaluation = EvaluationUpdateNew()

def perform_evaluation(model=None, tokenizer=None, text=None, starting_point=1):
    model.eval()
    batch_inputs = tokenizer.get_text_batch_encoding_for_loss(text=text) #for_loss
    #print(batch_inputs)
    with torch.no_grad():
        actual_output = model.forward_with_loss_simple( 
            #model.forward_with_loss_parallelized_efficient( #model.forward_with_loss_parallelized_efficient( #model.forward_with_loss_parallelized(
            batch_inputs=batch_inputs,
            compute_accuracy=True,
            return_dict=True,
            starting_point=0
        )
        print(actual_output)

        outs, pretokenized_score, _ = actual_output

    # memo_input = tokenizer.get_text_batch_encoding(text=text)
    # with torch.no_grad():
    #     pretokenized_score = evaluation.check_pretokenized(
    #         model=model,
    #         tokenizer=tokenizer,
    #         input_ids=memo_input['input_ids'],
    #         starting_point=starting_point
    #     )
    loss = outs['loss']
    del outs
    model.train()
    torch.cuda.empty_cache()
    return dict(
        out_loss=loss,
        token_accuracy=pretokenized_score
    )

In [13]:
# results_1 = perform_evaluation(
#     model=model,
#     tokenizer=tokenizer,
#     text=[my_first_text]*1
# )
# print(results_1)

# results_2 = perform_evaluation(
#     model=model,
#     tokenizer=tokenizer,
#     text=[my_second_text]*1
# )
# print(results_2)


In [14]:
memo_input_1#['input_ids'].shape

{'input_ids': tensor([[  0,   0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288,
          891, 298]]),
 'labels': tensor([[  0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288, 891,
          298, 278]])}

In [15]:
# memorize input
# model.memorize_text(memo_input)
model.memorize_text(memo_input_1)

In [16]:
tokenizer.get_text_batch_encoding_for_loss([my_first_text]*1)#['input_ids'].shape

{'input_ids': tensor([[  0,   0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288,
          891, 298]]),
 'labels': tensor([[-100, -100, -100, -100, -100,   66,  270,  260,  277,  299,  269,  305,
           288,  891,  298,  278]])}

In [17]:
memo_input_1['labels'], [ 66,  66,  66,  66,  66,  66, 270, 260, 277, 299, 269, 305, 288, 891,
         298, 278]

(tensor([[  0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288, 891,
          298, 278]]),
 [66, 66, 66, 66, 66, 66, 270, 260, 277, 299, 269, 305, 288, 891, 298, 278])

In [18]:
model.memo.encoder.weight[tokenizer.pad_token_id], model.memo.output_encoder.weight[tokenizer.pad_token_id]

(tensor([0.0221, 0.0221, 0.0221,  ..., 0.0221, 0.0221, 0.0221], device='cuda:0',
        grad_fn=<SelectBackward0>),
 tensor([0., 0., 0.,  ..., 0., 0., 0.], device='cuda:0',
        grad_fn=<SelectBackward0>))

In [19]:
o = model.retrieve(memo_input_1['input_ids'])
o.logits.shape#argmax(dim=-1)#.shape

sequence_encoding.shape torch.Size([1, 16, 2048])
CorrelationMatrixMemory(in_features=2048, out_features=2048)
layered_out_token.shape torch.Size([1, 16, 2048])
Adding layered_out_token torch.Size([1, 16, 2048])
sequence_representation.shape torch.Size([1, 16, 2048])
outputs['layered_out_token'].shape torch.Size([1, 16, 2048])
(tensor([[ 66,  66,  66,  66,  66,  66, 270, 260, 277, 299, 269, 305, 288, 891,
         298, 278]], device='cuda:0'), tensor([[ 9.5774,  9.5774,  9.5774,  9.5774,  9.5774,  9.5774, 11.7030, 11.7018,
         11.8902, 11.6421, 11.6439, 11.7334, 11.7461, 10.9460, 11.6050, 12.1288]],
       device='cuda:0', grad_fn=<MaxBackward0>))
residual_stream.shape torch.Size([1, 16, 2048])
sequence_encoding.shape torch.Size([1, 16, 2048])
CorrelationMatrixMemory(in_features=2048, out_features=2048)
layered_out_token.shape torch.Size([1, 16, 2048])
Adding layered_out_token torch.Size([1, 16, 2048])
sequence_representation.shape torch.Size([1, 16, 2048])
outputs['layered_out_to

torch.Size([1, 16, 50277])

In [20]:
o.logits.max(dim=-1).indices == memo_input_1['labels'].to('cuda'), memo_input_1['labels']

(tensor([[False, False, False, False, False,  True,  True,  True,  True,  True,
           True,  True,  True,  True,  True,  True]], device='cuda:0'),
 tensor([[  0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288, 891,
          298, 278]]))

In [21]:
# post-edit evaluation
# results = perform_evaluation(
#     model=model,
#     tokenizer=tokenizer,
#     text=first_text*8
# )
# print(results)

# results_2 = perform_evaluation(
#     model=model,
#     tokenizer=tokenizer,
#     text=first_text,
#     starting_point=None
# )
# print(results_2)

results_1 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_first_text]*1
)
print(results_1)

results_2 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_second_text]*1
)
print(results_2)

sequence_encoding.shape torch.Size([1, 16, 2048])
CorrelationMatrixMemory(in_features=2048, out_features=2048)
layered_out_token.shape torch.Size([1, 16, 2048])
Adding layered_out_token torch.Size([1, 16, 2048])
sequence_representation.shape torch.Size([1, 16, 2048])
outputs['layered_out_token'].shape torch.Size([1, 16, 2048])
(tensor([[ 66,  66,  66,  66,  66,  66, 270, 260, 277, 299, 269, 305, 288, 891,
         298, 278]], device='cuda:0'), tensor([[ 9.5774,  9.5774,  9.5774,  9.5774,  9.5774,  9.5774, 11.7030, 11.7018,
         11.8902, 11.6421, 11.6439, 11.7334, 11.7461, 10.9460, 11.6050, 12.1288]],
       device='cuda:0'))
residual_stream.shape torch.Size([1, 16, 2048])
sequence_encoding.shape torch.Size([1, 16, 2048])
CorrelationMatrixMemory(in_features=2048, out_features=2048)
layered_out_token.shape torch.Size([1, 16, 2048])
Adding layered_out_token torch.Size([1, 16, 2048])
sequence_representation.shape torch.Size([1, 16, 2048])
outputs['layered_out_token'].shape torch.Size([

In [22]:
model.memorize_text(memo_input_2)

In [23]:
results_1 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_first_text]*1
)
print(results_1)

results_2 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_second_text]*1
)
print(results_2)

sequence_encoding.shape torch.Size([1, 16, 2048])
CorrelationMatrixMemory(in_features=2048, out_features=2048)
layered_out_token.shape torch.Size([1, 16, 2048])
Adding layered_out_token torch.Size([1, 16, 2048])
sequence_representation.shape torch.Size([1, 16, 2048])
outputs['layered_out_token'].shape torch.Size([1, 16, 2048])
(tensor([[ 66,  66,  66,  66,  66,  66, 270, 260, 277, 299, 269, 305, 288, 891,
         298, 278]], device='cuda:0'), tensor([[ 7.8013,  7.8013,  7.8013,  7.8013,  7.8013,  7.8013, 10.9432, 10.6044,
         10.8963, 10.6596, 10.6105, 10.7492, 10.6846,  9.9857, 10.3769, 11.1668]],
       device='cuda:0'))
residual_stream.shape torch.Size([1, 16, 2048])
sequence_encoding.shape torch.Size([1, 16, 2048])
CorrelationMatrixMemory(in_features=2048, out_features=2048)
layered_out_token.shape torch.Size([1, 16, 2048])
Adding layered_out_token torch.Size([1, 16, 2048])
sequence_representation.shape torch.Size([1, 16, 2048])
outputs['layered_out_token'].shape torch.Size([

In [24]:
memo_input_2['labels']

tensor([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  75, 465, 278, 247,
         270, 260]])

In [31]:
import torch
from MeMoHF.modelling_memo_tokenizer import MeMoTokenizer
from MeMoHF.modelling_memo_configuration import MeMoConfig
from MeMoHF.modelling_memo import MeMoForCausalLM
from MeMoHF.evaluating_memo import Evaluation
from MeMoHF.utils import seed_everything

seed_everything(42)

# d, h, l = 1024, 4, 4
# chunk_length = 256

# d, h, l = 1024, 4, 2
# chunk_length = 16
d,h,l = 2048, 4, 3 #### increased #num_layers or chunk lenght --> lower performance
chunk_length = h ** l 

# Initializing a standard Tokenizer
max_length = chunk_length 
tokenizer = MeMoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", 
                                          padding_side='left', truncation_side='left', 
                                          model_max_length=max_length, head_number=h)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id



# Intializing Memo Configuration
config = MeMoConfig(vocab_size=len(tokenizer), #tokenizer.vocab_size, 
    hidden_size=d, 
    num_hidden_layers=l,
    num_attention_heads=h,
    chunk_length=chunk_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    compositionOp='prod',
    padding_seq_idx=tokenizer.pad_token_id,
    padding_vector_component_values= 1/(d**(1/2))  #0 # #new!
                    
)

# Initializing the Memo Model from the configuration

model = MeMoForCausalLM(config) 
model.training = True
model.train()

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    model.to('cuda')


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPTNeoXTokenizer'. 
The class this function is called from is 'MeMoTokenizer'.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Setting pad token and pad token id = <|endoftext|>, 0
Padding input tokens in MeMo architecture:  0.022097086912079608
MeMoEmbedding 0.022097086912079608
MeMoEmbedding 0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0.022097086912079608
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0.022097086912079608
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
GPU: NVIDIA RTX A6000 is available.


In [32]:
from datasets import Dataset, DatasetDict, Features, Value, load_dataset, load_from_disk, concatenate_datasets

data = load_from_disk('original_training_data/new_sample/mini/n=000020')
data

Dataset({
    features: ['text'],
    num_rows: 20
})

In [33]:
memo_input_3 = tokenizer.get_text_batch_encoding([data[0]['text']]*1)
model.memorize_text(memo_input_3)

Token indices sequence length is longer than the specified maximum sequence length for this model (233 > 65). Running this sequence through the model will result in indexing errors


In [34]:
## too much padding added??
a = tokenizer.get_text_batch_encoding_for_loss([data[1]['text']])['input_ids']
a, a.shape

(tensor([[    0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,    40, 14399,  1321, 36029,   313,  6448,   818,
           4223,  8441,    10,   310,   247,  5112,  5702, 31926,   665,  7120,
            323,  4009,  2083,  1914,  5210, 16004,    13,   327, 10119,   432,
           6365,    91,    13,   347,   247,   259,  4940,    15,   187,   187,
           4941,   187,   187,  1413,    27, 14316,   952,   187,  1413,    27,
          12487, 10782,   187,  1413],
         [43421,   432, 19548,  4415,   313,  7955,  2688,    10,   187,  1413,
             27,  8836,   432,   596,   279,  6169,   187,  1413,    27, 28521,
           5842, 12916,   398,   187,  1413,    27, 21258, 23236,   187,  1413,
             27,  6739,  6365,    91,  3773,   187,  1413,    27,    53,  2108,
          16004,  3773,   187,  1413,    27,  2838,  2083,  1914,  5210, 16004,
           3773,   187,  1413,    27,    45, 14406,   337,  3773,   187,  1413,
 

In [35]:
memo_input_4 = tokenizer.get_text_batch_encoding([data[1]['text']]*1)
model.memorize_text(memo_input_4)

In [36]:
print("After memorizing 3 and 4")

results_1 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_first_text]*1
)
print(results_1)
print("*"*80)

results_2 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[data[0]['text']]*1
)
print(results_2)
print("*"*80)

results_3 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[data[1]['text']]*1
)
print(results_3)
print("*"*80)


After memorizing 3 and 4
sequence_encoding.shape torch.Size([1, 64, 2048])
CorrelationMatrixMemory(in_features=2048, out_features=2048)
layered_out_token.shape torch.Size([1, 64, 2048])
Adding layered_out_token torch.Size([1, 64, 2048])
sequence_representation.shape torch.Size([1, 64, 2048])
outputs['layered_out_token'].shape torch.Size([1, 64, 2048])
(tensor([[   40,    40,    40,    40,    40,    40,    40,    40,    40,    40,
            40,    40,    40,    40,    40,    40,    40,    40,    40,    40,
            40,    40,    40,    40,    40,    40,    40,    40,    40,    40,
            40,    40,    40,    40,    40,    40,    40,    40,    40,    40,
            40,    40,    40,    40,    40,    40,    40,    40,    40,    40,
            40,    40,    40,    40, 25287,  5852,   187,   187,   285,  1413,
           187,    27,    13,  1413]], device='cuda:0'), tensor([[3.1727, 3.1727, 3.1727, 3.1727, 3.1727, 3.1727, 3.1727, 3.1727, 3.1727,
         3.1727, 3.1727, 3.1727, 

In [ ]:
# a = a + 1

In [ ]:
model.save_pretrained('memo_example')
tokenizer.save_pretrained('memo_example')

model2 = MeMoForCausalLM.from_pretrained('memo_example') #device_map='auto')
model2.to('cuda')
model2.train()
tokenizer2 = MeMoTokenizer.from_pretrained('memo_example')
model2

In [ ]:
print(model.memo.device)
print(model2.memo.device)

In [ ]:
# results = perform_evaluation(
#     model=model2,
#     tokenizer=tokenizer,
#     text=first_text*8
# )
# print(results)

results_1 = perform_evaluation(
    model=model2,
    tokenizer=tokenizer,
    text=[my_first_text]*1
)
print(results_1)

results_2 = perform_evaluation(
    model=model2,
    tokenizer=tokenizer,
    text=[my_second_text]*1
)
print(results_2)

In [ ]:
# results = perform_evaluation(
#     model=model2,
#     tokenizer=tokenizer2,
#     text=first_text*1
# )
# print(results)

In [ ]:
model2.memorize_text(memo_input_2)

results_1 = perform_evaluation(
    model=model2,
    tokenizer=tokenizer,
    text=[my_first_text]*1
)
print(results_1)

results_2 = perform_evaluation(
    model=model2,
    tokenizer=tokenizer,
    text=[my_second_text]*1
)
print(results_2)

In [ ]:
model2.forget_text(memo_input_2)

results_1 = perform_evaluation(
    model=model2,
    tokenizer=tokenizer,
    text=[my_first_text]*1
)
print(results_1)

results_2 = perform_evaluation(
    model=model2,
    tokenizer=tokenizer,
    text=[my_second_text]*1
)
print(results_2)

In [ ]:
# print(torch.exp(results['out_loss']))
# print(results)

In [ ]:
tokenizer.pad_token_type_id

In [ ]:
exit()